# Zimbabwe VACS 2017 — Covariate Codebook (Step 2)

Compiles the **covariate codebook** for Zimbabwe 2017 using the standard 5-column format:
**Category**, **Variable**, **Type**, **Format**, **Questions**.

**Data sources:**
- `.dta` PUD: `ZIMBABWE_VACS_2017_PUD.dta` — **single file** with `sex` column (1=male, 2=female)
- Codebook `.xlsx`: `Respondent_Codebook.xlsx`, `HOH_Codebook.xlsx` (no codebook PDFs for Zimbabwe)
- Questionnaire PDFs: `Female_RespondentQuestionnaire.pdf`, `Male_RespondentQuestionnaire.pdf`, `HOHQuestionnaire.pdf`

**Flow:** §1 Load & auto-scan → §2 Researcher-owned `RAW_MAP` → §3 Enhance from xlsx codebooks → §4 Final codebook DataFrame + TSV.

In [ ]:
from pathlib import Path
import sys
from IPython.display import display

import openpyxl
import pandas as pd
import pyreadstat

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

ZIM_DIR = ROOT / "data" / "raw" / "Zimbabwe Stata"
PUD_PATH = ZIM_DIR / "ZIMBABWE_VACS_2017_PUD.dta"

RESP_CODEBOOK = ZIM_DIR / "ZIMBABWE_VACS_2017_Respondent_Codebook.xlsx"
HOH_CODEBOOK = ZIM_DIR / "ZIMBABWE_VACS_2017_HOH_Codebook.xlsx"

## 1. Load data & auto-scan

Single PUD with both sexes. Scan Stata labels for covariate candidates, then parse xlsx codebooks for question text and response options.

In [ ]:
df, meta = pyreadstat.read_dta(PUD_PATH)
print(f"PUD: {df.shape[0]:,} rows × {df.shape[1]:,} cols")
print(f"sex distribution: {df['sex'].value_counts(dropna=False).to_dict()}")

In [ ]:
from utils.covariates import search_dta_for_covariates

candidates = search_dta_for_covariates(df, meta)
with pd.option_context("display.max_colwidth", 80, "display.width", 220, "display.max_rows", 50):
    display(candidates)

In [ ]:
def parse_xlsx_codebook(xlsx_path):
    """Parse a Zimbabwe-style xlsx codebook into {var_name: {question, responses}}."""
    wb = openpyxl.load_workbook(xlsx_path)
    ws = wb.active
    entries = {}
    HEADER_WORDS = {"Skip", "Frequency", "Percent"}
    i = 1
    while i <= ws.max_row:
        cell_a = ws.cell(i, 1).value
        if cell_a is None:
            i += 1
            continue
        val = str(cell_a).strip()
        # Variable name row: check cols B–D for header keywords
        is_var_row = any(
            ws.cell(i, c).value and str(ws.cell(i, c).value).strip() in HEADER_WORDS
            for c in range(2, 5)
        )
        if is_var_row:
            var_name = val.lower()
            q_text = str(ws.cell(i - 1, 1).value or "").strip()
            responses = []
            j = i + 1
            while j <= ws.max_row:
                rv = ws.cell(j, 1).value
                if rv is None or str(rv).strip() == "" or str(rv).strip() == "Total":
                    break
                responses.append(str(rv).strip())
                j += 1
            entries[var_name] = {"question": q_text, "responses": responses}
        i += 1
    return entries

resp_cb = parse_xlsx_codebook(RESP_CODEBOOK)
hoh_cb = parse_xlsx_codebook(HOH_CODEBOOK)
print(f"Respondent codebook entries: {len(resp_cb)}")
print(f"HOH codebook entries:        {len(hoh_cb)}")

## 2. Researcher-owned covariate mapping

Zimbabwe uses a **single PUD** with a `sex` column — no male/female file split. Variable names are lowercase. The `variable` column uses the Stata column name directly (no `Q#; F#` notation needed).

Helper to look up question text and format from the parsed xlsx codebooks:

In [ ]:
def format_responses(resp_list):
    """Convert xlsx response list to codebook format string."""
    if not resp_list:
        return ""
    parts = []
    for r in resp_list:
        r = r.strip()
        # Normalize '1 =YES' -> '1-YES', '1=YES' -> '1-YES'
        r = r.replace(" =", "-").replace("=", "-")
        # Skip plain numeric values (age/count ranges)
        try:
            int(r)
            continue
        except ValueError:
            pass
        parts.append(r)
    return ", ".join(parts)


def lookup(var_name):
    """Look up question text and format from xlsx codebooks."""
    key = var_name.lower()
    e = resp_cb.get(key) or hoh_cb.get(key)
    if e:
        return e["question"], format_responses(e["responses"])
    return "", ""

In [ ]:
from utils.covariates import classify_variable_type

# --- Researcher-owned mapping ---
# (category, variable_notation, stata_var, type_override)
# stata_var: the .dta column name used for type classification + xlsx lookup
# type_override: set manually when heuristic won't work

RAW_MAP = [
    # Respondent-level covariates
    ("Sex",                      "sex",        "sex",     "categorical"),
    ("Age",                      "q2",         "q2",      "numerical"),
    ("Highest Education Level",  "q5",         "q5",      ""),
    ("Enough Money for\u2026",   "NA",         "",        "NA"),
    ("Lives with biological mom","q17",        "q17",     ""),
    ("Lives with biological dad","q23",        "q23",     ""),
    ("Lives in foster care",     "NA",         "",        "NA"),
    ("Ever moved",               "NA",         "",        "NA"),
    ("Ever Married",             "q31_rc",     "q31_rc",  ""),
    ("Disability",               "NA",         "",        "NA"),
    ("Community Trust",          "q46",        "q46",     ""),
    ("Community Safety",         "q47",        "q47",     ""),
    ("Supportive friends",       "q7",         "q7",      ""),
    ("Engage in work for pay in last 12 months",
                                 "q13",        "q13",     ""),
    ("Drank alcohol in last 30 days",
                                 "q1201",      "q1201",   ""),
    ("Smoke Cigarettes in last 30 days",
                                 "q1202",      "q1202",   ""),
    ("Mental Health",            "q1204a",     "q1204a",  "categorical"),
    ("Mental Health",            "q1204b",     "q1204b",  "categorical"),
    ("Mental Health",            "q1204c",     "q1204c",  "categorical"),
    ("Mental Health",            "q1204e",     "q1204e",  "categorical"),
    ("Mental Health",            "q1204f",     "q1204f",  "categorical"),
    # Household-level covariates (HOH questionnaire)
    ("Main source of drinking water (HH)",
                                 "h4",         "h4",      ""),
    ("Flush toilet (HH)",        "h12",        "h12",     ""),
    ("Shared HH",                "h14",        "h14",     ""),
    ("Electricity (HH)",         "h18a",       "h18a",    ""),
    ("Dwelling floor (HH)",      "h27",        "h27",     ""),
    ("Roof type (HH)",           "h29",        "h29",     ""),
    ("Wall material (HH)",       "h30",        "h30",     ""),
    ("# rooms in household (HH)","h31",        "h31",     ""),
    ("# rooms in household (HH)","h32",        "h32",     ""),
]

## 3. Enhance from xlsx codebooks

For each entry, look up question text and response options from the parsed xlsx codebooks. Use `classify_variable_type()` for Type when not manually overridden.

In [ ]:
entries = []

# Manual overrides for variables not in xlsx codebooks
MANUAL_QF = {
    "sex": ("Sex of respondent", "1-Male, 2-Female"),
}

for cat, var_notation, stata_var, type_ov in RAW_MAP:
    if var_notation == "NA":
        entries.append({
            "category": cat, "variable": "NA",
            "type": "NA", "format": "", "question": "",
        })
        continue

    # Look up from xlsx codebooks, with manual override fallback
    if stata_var in MANUAL_QF:
        q_text, q_format = MANUAL_QF[stata_var]
    else:
        q_text, q_format = lookup(stata_var)
        # For q31_rc, the codebook has q31 (original); try both
        if not q_text and "_rc" in stata_var:
            q_text, q_format = lookup(stata_var.replace("_rc", ""))

    # Determine type
    if type_ov:
        var_type = type_ov
    elif stata_var and stata_var in df.columns:
        var_type = classify_variable_type(df[stata_var])
    else:
        var_type = ""

    entries.append({
        "category": cat,
        "variable": var_notation,
        "type": var_type,
        "format": q_format,
        "question": q_text,
    })

print(f"Entries built: {len(entries)}")

## 4. Harmonized covariate codebook

Build the 5-column DataFrame and produce TSV for Excel paste.

In [ ]:
from utils.covariates import build_covariate_codebook_df, covariate_codebook_to_tsv

codebook_df = build_covariate_codebook_df(entries)

with pd.option_context(
    "display.max_colwidth", 80,
    "display.width", 220,
    "display.max_rows", 40,
):
    display(codebook_df)

In [ ]:
from docx import Document

OUT_PATH = ROOT / "data" / "processed" / "jcx_zimbabweCovariateCodebook_V2.docx"

doc = Document()
table = doc.add_table(rows=1 + len(codebook_df), cols=len(codebook_df.columns))
for j, col_name in enumerate(codebook_df.columns):
    table.rows[0].cells[j].text = col_name.title()
for i, (_, row) in enumerate(codebook_df.iterrows()):
    for j, col_name in enumerate(codebook_df.columns):
        table.rows[i + 1].cells[j].text = str(row[col_name]) if pd.notna(row[col_name]) else ""

# Merge consecutive Category cells that share the same value
cats = codebook_df["category"].tolist()
start = 0
while start < len(cats):
    end = start
    while end + 1 < len(cats) and cats[end + 1] == cats[start]:
        end += 1
    if end > start:
        table.cell(start + 1, 0).merge(table.cell(end + 1, 0))
    start = end + 1

doc.save(str(OUT_PATH))
print(f"Saved → {OUT_PATH}")